# 2-bit quantizer sample-size experiment

This notebook follows the Topic 06 sample-size workflow: choose confidence and power, estimate the standard deviation with a pilot run, then calculate how many paired observations are needed.

The three active algorithms are Leech shell-13, uniform scalar 2-bit, and QTIP bitshift K=2. E8 and RaBitQ are intentionally excluded. The final analysis uses multiple-sample tests first, followed by planned Bonferroni-corrected post-hoc paired comparisons.

In [ ]:
from __future__ import annotations

import contextlib
import importlib.util
import io
import math
import subprocess
import sys
import time
import types
from dataclasses import dataclass
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    TORCH_AVAILABLE = True
    TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc
    TORCH_DEVICE = None

try:
    import pandas as pd
except Exception:
    pd = None

print(f"root: {ROOT}")
print(f"torch available: {TORCH_AVAILABLE}")
print(f"torch device: {TORCH_DEVICE}")


In [ ]:
# Experiment configuration.
SEED = 0
DIM = 24
TARGET_BITS_PER_DIM = 2.0

# Algorithms: Leech shell-13, uniform scalar 2-bit, and QTIP bitshift K=2.
LEECH_MAX_SHELL = 13
LEECH_ANN_TOP_SHELLS = 0
QTIP_REPO = ROOT / "third_party" / "qtip"
AUTO_CLONE_QTIP = True
QTIP_GIT_URL = "https://github.com/Cornell-RelaxML/qtip.git"
QTIP_CONFIG = dict(L=8, K=int(TARGET_BITS_PER_DIM), V=1, tlut_bits=8, decode_mode="quantlut")

# No external/global scale in this experiment.
USE_SHARED_SCALE = False
SCALE_BITS = 0
SCALE_GRANULARITY = "none"
PER_VECTOR_SCALAR_BITS = 0

# Sample size calculation settings from the lecture: alpha, power, SD, and minimum interesting effect.
# Quality samples are vectors. Timing samples are repeated benchmark trials on a fixed batch.
PILOT_QUALITY_N = 256
PILOT_TIMING_REPEATS = 30
TIMING_BATCH_N = 1024
TIMING_WARMUP_REPEATS = 5
TIMING_RANDOMIZE_ORDER = True

ALPHA = 0.05
POWER = 0.80
TWO_SIDED = True
# Controls post-hoc Type-I error. "all_metrics" controls all 3 metrics x 3 comparisons together.
# Use "per_metric" to control each metric family separately.
FWER_SCOPE = "all_metrics"
N_METRICS_FOR_FWER = 3

# Minimum interesting effects. Adjust these to match the claim you want to make.
MIN_INTERESTING_SQNR_BITS_DIFF = 0.10
MIN_INTERESTING_QUANTIZE_SECONDS = 0.001
MIN_INTERESTING_DEQUANTIZE_SECONDS = 0.00005

rng = np.random.default_rng(SEED)


In [ ]:
@dataclass
class QuantizerReport:
    method: str
    rate_bits_per_dim: float
    mse: float
    rmse: float
    error_std: float
    squared_error_std: float
    sqnr_bits: float
    scale: float
    seconds: float
    quantize_seconds: float
    dequantize_seconds: float
    metadata: dict


def mse_to_sqnr_bits(mse: float) -> float:
    return float("inf") if mse == 0 else -0.5 * math.log2(mse)


def reconstruction_error_stats(x: np.ndarray, recon: np.ndarray) -> dict[str, float]:
    error = x - recon
    squared_error = error ** 2
    mse = float(np.mean(squared_error))
    return {
        "mse": mse,
        "rmse": float(math.sqrt(mse)),
        "error_std": float(np.std(error)),
        "squared_error_std": float(np.std(squared_error)),
    }


def synchronize_for_timing():
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.synchronize()


def scale_overhead_bpd(x: np.ndarray, scale_bits: int, granularity: str) -> float:
    if scale_bits <= 0:
        return 0.0
    if granularity == "global":
        return float(scale_bits) / float(x.size)
    if granularity == "per_vector":
        return float(scale_bits) / float(x.shape[-1])
    raise ValueError(f"unknown scale granularity: {granularity}")


def optimize_global_scale(x: np.ndarray, quantize_normalized, scales: np.ndarray):
    best_scale = float(scales[0])
    best_mse = float("inf")
    best_recon = None
    for scale in scales:
        recon, mse = quantize_at_scale(x, quantize_normalized, float(scale))
        if mse < best_mse:
            best_scale = float(scale)
            best_mse = mse
            best_recon = recon
    return best_scale, best_mse, best_recon


def quantize_at_scale(x: np.ndarray, quantize_normalized, scale: float):
    recon = float(scale) * quantize_normalized(x / float(scale))
    stats = reconstruction_error_stats(x, recon)
    return recon, stats["mse"]


def choose_shared_scale(x: np.ndarray, quantizers, scales: np.ndarray):
    best_scale = float(scales[0])
    best_mean_mse = float("inf")
    for scale in scales:
        mses = [quantize_at_scale(x, item["quantize"], float(scale))[1] for item in quantizers]
        mean_mse = float(np.mean(mses))
        if mean_mse < best_mean_mse:
            best_scale = float(scale)
            best_mean_mse = mean_mse
    return best_scale, best_mean_mse


def run_scaled_quantizer(method: str, x: np.ndarray, rate: float, quantize_normalized, scales, metadata=None, shared_scale=None):
    metadata = {} if metadata is None else dict(metadata)
    start = time.perf_counter()
    if shared_scale is None:
        scale, mse, recon = optimize_global_scale(x, quantize_normalized, np.asarray(scales, dtype=np.float32))
        metadata.setdefault("scale_mode", "per_method_optimized")
    else:
        scale = float(shared_scale)
        recon, mse = quantize_at_scale(x, quantize_normalized, scale)
        metadata.setdefault("scale_mode", "shared")
    seconds = time.perf_counter() - start
    stats = reconstruction_error_stats(x, recon)
    report = QuantizerReport(
        method=method,
        rate_bits_per_dim=float(rate),
        mse=stats["mse"],
        rmse=stats["rmse"],
        error_std=stats["error_std"],
        squared_error_std=stats["squared_error_std"],
        sqnr_bits=mse_to_sqnr_bits(stats["mse"]),
        scale=float(scale),
        seconds=float(seconds),
        quantize_seconds=float(seconds),
        dequantize_seconds=0.0,
        metadata=metadata,
    )
    return report, recon


def run_split_scaled_quantizer(method: str, x: np.ndarray, rate: float, encode_normalized, decode_normalized, metadata=None, shared_scale=1.0):
    metadata = {} if metadata is None else dict(metadata)
    metadata.setdefault("scale_mode", "shared")
    scale = float(shared_scale)
    normalized = x / scale

    synchronize_for_timing()
    quantize_start = time.perf_counter()
    codes = encode_normalized(normalized)
    synchronize_for_timing()
    quantize_seconds = time.perf_counter() - quantize_start

    synchronize_for_timing()
    dequantize_start = time.perf_counter()
    recon_normalized = decode_normalized(codes)
    synchronize_for_timing()
    dequantize_seconds = time.perf_counter() - dequantize_start

    recon = scale * recon_normalized
    stats = reconstruction_error_stats(x, recon)
    report = QuantizerReport(
        method=method,
        rate_bits_per_dim=float(rate),
        mse=stats["mse"],
        rmse=stats["rmse"],
        error_std=stats["error_std"],
        squared_error_std=stats["squared_error_std"],
        sqnr_bits=mse_to_sqnr_bits(stats["mse"]),
        scale=scale,
        seconds=float(quantize_seconds + dequantize_seconds),
        quantize_seconds=float(quantize_seconds),
        dequantize_seconds=float(dequantize_seconds),
        metadata=metadata,
    )
    return report, recon


def reports_table(reports):
    rows = [
        {
            "method": r.method,
            "rate_bpd": r.rate_bits_per_dim,
            "side_info_bpd": float(r.metadata.get("side_info_bpd", 0.0)),
            "effective_rate_bpd": r.rate_bits_per_dim + float(r.metadata.get("side_info_bpd", 0.0)),
            "target_delta_bpd": r.rate_bits_per_dim + float(r.metadata.get("side_info_bpd", 0.0)) - TARGET_BITS_PER_DIM,
            "mse": r.mse,
            "rmse": r.rmse,
            "error_std": r.error_std,
            "squared_error_std": r.squared_error_std,
            "sqnr_bits": r.sqnr_bits,
            "scale": r.scale,
            "seconds": r.seconds,
            "quantize_seconds": r.quantize_seconds,
            "dequantize_seconds": r.dequantize_seconds,
            "metadata": r.metadata,
        }
        for r in reports
    ]
    if pd is not None:
        return pd.DataFrame(rows).sort_values(["mse", "seconds"]).reset_index(drop=True)
    return rows


## Leech lattice vector quantizer

This wraps the local `leech_lattice_vector_quantizer` module. If CUDA is available, it uses the batched GPU implementation; otherwise it falls back to the CPU implementation. The current CPU implementation prints the winning codeword during quantization, so the wrapper suppresses that output for table runs.

In [ ]:
class LeechNotebookQuantizer:
    def __init__(self, max_shell: int, ann_top_shells: int = 0):
        self.max_shell = max_shell
        self.ann_top_shells = int(ann_top_shells)
        self.backend = "cpu"
        self.device = "cpu"

        if TORCH_AVAILABLE and torch.cuda.is_available():
            try:
                from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer_gpu import LeechLatticeVectorQuantizerGpu

                self.quantizer = LeechLatticeVectorQuantizerGpu(max_shell=max_shell, verbose=False, device="cuda")
                self.quantizer._quantize_cuda_metadata()["ann_top_shells"] = self.ann_top_shells
                # Trigger NVRTC compilation here so setup errors are reported before the benchmark loop.
                warm_x = torch.zeros((1, 24), dtype=torch.float32, device=self.quantizer.device)
                warm_idx = self.quantizer.quantize(warm_x)
                _ = self.quantizer.dequantize(warm_idx, check_bounds=False)
                torch.cuda.synchronize()
                self.backend = "gpu"
                self.device = str(self.quantizer.device)
            except Exception as exc:
                print(f"Leech GPU unavailable, falling back to CPU: {type(exc).__name__}: {exc}")
                from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer import LeechLatticeVectorQuantizer

                self.quantizer = LeechLatticeVectorQuantizer(max_shell=max_shell, verbose=False)
        else:
            from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer import LeechLatticeVectorQuantizer

            self.quantizer = LeechLatticeVectorQuantizer(max_shell=max_shell, verbose=False)
        self.rate = self.quantizer.shape_bits / 24.0

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if samples.shape[-1] != 24:
            raise ValueError("Leech quantizer expects 24-D rows")
        if self.backend == "gpu":
            x = torch.as_tensor(samples, dtype=torch.float32, device=self.quantizer.device)
            with torch.no_grad():
                return self.quantizer.quantize(x)

        indices = np.empty(samples.shape[:-1], dtype=np.int64)
        for i, row in enumerate(samples):
            with contextlib.redirect_stdout(io.StringIO()):
                indices[i] = self.quantizer.quantize(row)
        return indices

    def decode_normalized(self, indices) -> np.ndarray:
        if self.backend == "gpu":
            with torch.no_grad():
                recon = self.quantizer.dequantize(indices, check_bounds=False)
            return recon.detach().cpu().numpy().astype(np.float32)

        out = np.empty((*np.asarray(indices).shape, 24), dtype=np.float32)
        for i, idx in enumerate(np.asarray(indices).reshape(-1)):
            out.reshape(-1, 24)[i] = np.asarray(self.quantizer.dequantize(int(idx)), dtype=np.float32)
        return out


## QTIP bitshift quantizer

The QTIP repository imports compiled kernel bindings at package import time. For this notebook's codebook-only test, the loader below installs minimal stubs and imports `lib/codebook/bitshift.py` directly. Install `torch` first; the notebook can clone `Cornell-RelaxML/qtip` into `third_party/qtip` if missing.

In [ ]:
def ensure_qtip_repo(repo_path: Path, auto_clone: bool = True) -> Path | None:
    if repo_path.exists():
        return repo_path
    if not auto_clone:
        return None
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", QTIP_GIT_URL, str(repo_path)], check=True)
    return repo_path


def load_qtip_bitshift_module(qtip_repo: Path):
    if not TORCH_AVAILABLE:
        raise RuntimeError(f"torch is required for QTIP: {TORCH_IMPORT_ERROR}")

    sys.modules.setdefault("lib", types.ModuleType("lib"))

    codebook_stub = types.ModuleType("lib.codebook")
    codebook_stub.kdict = {}
    sys.modules["lib.codebook"] = codebook_stub

    sys.modules.setdefault("lib.utils", types.ModuleType("lib.utils"))

    kernel_check = types.ModuleType("lib.utils.kernel_check")
    kernel_check.has_kernel = lambda *args, **kwargs: False
    sys.modules["lib.utils.kernel_check"] = kernel_check

    kernel_decompress = types.ModuleType("lib.utils.kernel_decompress")

    def _decode_compressed_not_used(*args, **kwargs):
        raise RuntimeError("kernel decompression is not used in this codebook-only notebook test")

    kernel_decompress.decode_compressed = _decode_compressed_not_used
    sys.modules["lib.utils.kernel_decompress"] = kernel_decompress

    matmul_had = types.ModuleType("lib.utils.matmul_had")
    matmul_had.matmul_hadU_cuda = lambda *args, **kwargs: None
    matmul_had.matmul_hadUt_cuda = lambda *args, **kwargs: None
    sys.modules["lib.utils.matmul_had"] = matmul_had

    module_path = qtip_repo / "lib" / "codebook" / "bitshift.py"
    spec = importlib.util.spec_from_file_location("qtip_bitshift_notebook", module_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(module_path)

    # The codebook smoke test runs fine in eager mode. QTIP decorates one helper with
    # torch.compile, which requires a C++ compiler through Inductor even on CPU.
    # Temporarily disabling the decorator keeps this notebook portable.
    original_compile = torch.compile
    torch.compile = lambda fn=None, *args, **kwargs: (lambda f: f) if fn is None else fn
    try:
        spec.loader.exec_module(module)
    finally:
        torch.compile = original_compile
    return module


class QTIPNotebookQuantizer:
    def __init__(self, repo_path: Path, config: dict, device=None):
        module = load_qtip_bitshift_module(repo_path)
        self.config = dict(config)
        self.device = device if device is not None else torch.device("cpu")
        self.backend = "gpu" if self.device.type == "cuda" else "cpu"
        self.cb = module.bitshift_codebook(**self.config).to(self.device)
        self.cb.fakeinf = self.cb.fakeinf.to(self.device)
        self.rate = float(self.config["K"])

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        x = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            x_t = x.T.contiguous().to(torch.float16)
            T = x_t.shape[0]
            roll_x = torch.roll(x_t, T // (2 * self.cb.V) * self.cb.V, 0)
            states = self.cb.quantize_seq(roll_x, overlap=None)
            overlap = states[T // (2 * self.cb.V)] >> self.cb.K * self.cb.V
            states = self.cb.quantize_seq(x_t, overlap=overlap)
            states = states.T.contiguous().to(self.device)
        self.last_states = states
        return states

    def decode_normalized(self, states) -> np.ndarray:
        states = states.to(self.device) if torch.is_tensor(states) else torch.as_tensor(states, device=self.device)
        with torch.no_grad():
            states_t = states.T.contiguous()
            batch = states.shape[0]
            dim = states.shape[1] * self.cb.V
            recon = self.cb.recons(states_t).transpose(0, 1).reshape(dim, batch).T.contiguous()
        self.last_states = states
        return recon.detach().cpu().numpy().astype(np.float32)


## Uniform scalar quantizer

A clipped per-coordinate uniform scalar quantizer is included as a simple exact-rate baseline.

In [ ]:
class UniformScalarNotebookQuantizer:
    def __init__(self, bits: int, device=None):
        if bits < 1:
            raise ValueError("bits must be positive")
        self.bits = int(bits)
        self.levels = 1 << self.bits
        self.qmin = -(self.levels // 2)
        self.qmax = self.levels // 2 - 1
        self.rate = float(self.bits)
        self.device = device if TORCH_AVAILABLE and device is not None else None
        self.backend = "gpu" if self.device is not None and self.device.type == "cuda" else "cpu"

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if self.device is None:
            return np.clip(np.round(samples), self.qmin, self.qmax).astype(np.int16)
        x = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            return torch.clamp(torch.round(x), self.qmin, self.qmax).to(torch.int16)

    def decode_normalized(self, codes) -> np.ndarray:
        if torch is not None and torch.is_tensor(codes):
            return codes.to(torch.float32).detach().cpu().numpy()
        return np.asarray(codes, dtype=np.float32)


## Sample size calculation

This notebook calculates three sample sizes:

- Quality: number of paired vectors needed to compare per-vector SQNR bits.
- Quantize time: number of repeated benchmark trials needed to compare quantize time.
- Dequantize time: number of repeated benchmark trials needed to compare dequantize time.

The analysis layout follows the multiple-sample testing lecture: first use a global multiple-sample test for each metric, then use planned post-hoc comparisons to identify which algorithms differ. Because the observations are paired/repeated measures, the global tests are repeated-measures ANOVA and Friedman. The sample-size planning is still based on the planned post-hoc paired differences, with the alpha value corrected for multiple comparisons.

In [ ]:
from itertools import combinations

from scipy import stats


def normal_quantile(p: float) -> float:
    return float(stats.norm.ppf(float(p)))


def pilot_sample_size_for_sd(confidence: float = 0.95, relative_error: float = 0.25) -> int:
    # Topic 06 pilot-study approximation: n_pilot ~= 2 * (z_{alpha_n/2} / e_n)^2.
    alpha_n = 1.0 - float(confidence)
    z = normal_quantile(1.0 - alpha_n / 2.0)
    return int(math.ceil(2.0 * (z / float(relative_error)) ** 2))


def paired_t_power(n: int, sd_diff: float, delta: float, alpha: float, two_sided: bool = True) -> float:
    if n < 2 or sd_diff <= 0.0:
        return 0.0
    df = int(n) - 1
    ncp = math.sqrt(float(n)) * float(delta) / float(sd_diff)
    if two_sided:
        tcrit = float(stats.t.ppf(1.0 - alpha / 2.0, df))
        value = float(stats.nct.sf(tcrit, df, ncp) + stats.nct.cdf(-tcrit, df, ncp))
    else:
        tcrit = float(stats.t.ppf(1.0 - alpha, df))
        value = float(stats.nct.sf(tcrit, df, ncp))
    if not math.isfinite(value):
        return 1.0 if ncp > 0.0 else 0.0
    return min(1.0, max(0.0, value))


def paired_sample_size_t(sd_diff: float, delta: float, alpha: float, power: float, two_sided: bool = True) -> int:
    if delta <= 0.0:
        raise ValueError("delta must be positive")
    if sd_diff <= 0.0:
        return 2

    lo = 2
    hi = 2
    while paired_t_power(hi, sd_diff, delta, alpha, two_sided) < power:
        hi *= 2
        if hi > 10_000_000:
            raise RuntimeError("sample-size search exceeded 10,000,000 observations")

    while lo < hi:
        mid = (lo + hi) // 2
        if paired_t_power(mid, sd_diff, delta, alpha, two_sided) >= power:
            hi = mid
        else:
            lo = mid + 1
    return int(lo)


def per_vector_sqnr_bits(x: np.ndarray, recon: np.ndarray, eps: float = 1e-30) -> np.ndarray:
    x64 = x.astype(np.float64, copy=False)
    recon64 = recon.astype(np.float64, copy=False)
    signal_power = np.mean(x64 ** 2, axis=1)
    noise_power = np.mean((x64 - recon64) ** 2, axis=1)
    ratio = np.maximum(signal_power, eps) / np.maximum(noise_power, eps)
    return 0.5 * np.log2(ratio)


def make_quantizer_items():
    quantizer_device = TORCH_DEVICE if TORCH_AVAILABLE else None
    items = []

    leech = LeechNotebookQuantizer(LEECH_MAX_SHELL, ann_top_shells=LEECH_ANN_TOP_SHELLS)
    leech_mode = "exact" if leech.ann_top_shells <= 0 else f"top{leech.ann_top_shells}"
    items.append({
        "method": f"Leech shells <= {LEECH_MAX_SHELL} {leech_mode} ({leech.backend})",
        "rate": leech.rate,
        "encode": leech.encode_normalized,
        "decode": leech.decode_normalized,
        "metadata": {
            "target_bpd": TARGET_BITS_PER_DIM,
            "backend": leech.backend,
            "device": leech.device,
            "ann_top_shells": leech.ann_top_shells,
            "approximate": bool(leech.ann_top_shells > 0),
            "total_count": int(leech.quantizer.total_count),
            "shape_bits": int(leech.quantizer.shape_bits),
        },
    })

    uniform = UniformScalarNotebookQuantizer(bits=int(TARGET_BITS_PER_DIM), device=quantizer_device)
    items.append({
        "method": f"Uniform scalar 2-bit ({uniform.backend})",
        "rate": uniform.rate,
        "encode": uniform.encode_normalized,
        "decode": uniform.decode_normalized,
        "metadata": {
            "target_bpd": TARGET_BITS_PER_DIM,
            "bits": uniform.bits,
            "levels": uniform.levels,
            "backend": uniform.backend,
            "device": str(uniform.device),
        },
    })

    qtip_status = "not run"
    if TORCH_AVAILABLE:
        try:
            qtip_repo = ensure_qtip_repo(QTIP_REPO, auto_clone=AUTO_CLONE_QTIP)
            qtip = QTIPNotebookQuantizer(qtip_repo, QTIP_CONFIG, device=quantizer_device)
            items.append({
                "method": f"QTIP bitshift ({qtip.backend})",
                "rate": qtip.rate,
                "encode": qtip.encode_normalized,
                "decode": qtip.decode_normalized,
                "metadata": {
                    "target_bpd": TARGET_BITS_PER_DIM,
                    "backend": qtip.backend,
                    "device": str(qtip.device),
                    **QTIP_CONFIG,
                },
            })
            qtip_status = f"run ({qtip.backend}, {qtip.device})"
        except Exception as exc:
            qtip_status = f"skipped: {type(exc).__name__}: {exc}"
    else:
        qtip_status = f"skipped: torch unavailable: {TORCH_IMPORT_ERROR}"

    return items, qtip_status


def run_reconstruction(item: dict, x: np.ndarray):
    report, recon = run_split_scaled_quantizer(
        item["method"],
        x,
        item["rate"],
        item["encode"],
        item["decode"],
        metadata={
            "side_info_bpd": 0.0,
            "scale_mode": "none",
            "shared_scale_bits": 0,
            "shared_scale_granularity": "none",
            **item["metadata"],
        },
        shared_scale=1.0,
    )
    return report, recon, per_vector_sqnr_bits(x, recon)


def timed_call(fn, *args):
    synchronize_for_timing()
    start = time.perf_counter()
    result = fn(*args)
    synchronize_for_timing()
    return result, time.perf_counter() - start


def benchmark_timing_samples(items: list[dict], x: np.ndarray, repeats: int, warmup: int = 5):
    quantize_times = {item["method"]: [] for item in items}
    dequantize_times = {item["method"]: [] for item in items}
    decode_inputs = {}

    for item in items:
        codes = None
        for _ in range(warmup):
            codes = item["encode"](x)
            _ = item["decode"](codes)
        synchronize_for_timing()
        decode_inputs[item["method"]] = codes

    for _ in range(int(repeats)):
        order = list(range(len(items)))
        if TIMING_RANDOMIZE_ORDER:
            rng.shuffle(order)
        for idx in order:
            item = items[idx]
            _, seconds = timed_call(item["encode"], x)
            quantize_times[item["method"]].append(seconds)

        order = list(range(len(items)))
        if TIMING_RANDOMIZE_ORDER:
            rng.shuffle(order)
        for idx in order:
            item = items[idx]
            _, seconds = timed_call(item["decode"], decode_inputs[item["method"]])
            dequantize_times[item["method"]].append(seconds)

    return {
        "quantize_seconds": {name: np.asarray(values, dtype=np.float64) for name, values in quantize_times.items()},
        "dequantize_seconds": {name: np.asarray(values, dtype=np.float64) for name, values in dequantize_times.items()},
    }


def pairwise_alpha_for_scope(n_methods: int, alpha: float) -> tuple[float, int]:
    n_comparisons = math.comb(n_methods, 2) if n_methods >= 2 else 0
    if n_comparisons == 0:
        return float(alpha), 1
    metric_families = int(N_METRICS_FOR_FWER) if FWER_SCOPE == "all_metrics" else 1
    total_planned_comparisons = n_comparisons * metric_families
    return float(alpha) / float(total_planned_comparisons), total_planned_comparisons


def sample_matrix(samples_by_method: dict[str, np.ndarray]) -> tuple[list[str], np.ndarray]:
    names = list(samples_by_method)
    values = [np.asarray(samples_by_method[name], dtype=np.float64) for name in names]
    if not values:
        raise ValueError("samples_by_method cannot be empty")
    first_shape = values[0].shape
    if any(value.shape != first_shape for value in values):
        raise ValueError("all paired sample arrays must have the same shape")
    return names, np.column_stack(values)


def sample_size_table_from_samples(samples_by_method: dict[str, np.ndarray], alpha: float, power: float, delta: float, metric: str, sample_unit: str):
    names = list(samples_by_method)
    n_comparisons = math.comb(len(names), 2) if len(names) >= 2 else 0
    alpha_pair, total_planned_comparisons = pairwise_alpha_for_scope(len(names), alpha)
    rows = []
    for a, b in combinations(names, 2):
        a_values = np.asarray(samples_by_method[a], dtype=np.float64)
        b_values = np.asarray(samples_by_method[b], dtype=np.float64)
        if a_values.shape != b_values.shape:
            raise ValueError(f"paired samples for {a} and {b} must have the same shape")
        diff = a_values - b_values
        sd_diff = float(np.std(diff, ddof=1))
        mean_diff = float(np.mean(diff))
        required_n = paired_sample_size_t(sd_diff, delta, alpha_pair, power, TWO_SIDED)
        rows.append({
            "metric": metric,
            "sample_unit": sample_unit,
            "comparison": f"{a} vs {b}",
            "pilot_mean_diff": mean_diff,
            "pilot_sd_diff": sd_diff,
            "delta": float(delta),
            "alpha_pair": float(alpha_pair),
            "fwer_scope": FWER_SCOPE,
            "total_planned_comparisons": int(total_planned_comparisons),
            "power": float(power),
            "required_n": required_n,
            "achieved_power_at_required_n": paired_t_power(required_n, sd_diff, delta, alpha_pair, TWO_SIDED),
        })
    if pd is not None:
        return pd.DataFrame(rows).sort_values("required_n", ascending=False).reset_index(drop=True)
    return rows


def required_n_from_table(table) -> int:
    if pd is not None and hasattr(table, "__len__") and len(table) > 0:
        return int(table["required_n"].max())
    return max(row["required_n"] for row in table)


def add_planned_power(table, planned_n: int):
    if pd is None:
        rows = []
        for row in table:
            row = dict(row)
            planned_power = paired_t_power(
                int(planned_n),
                float(row["pilot_sd_diff"]),
                float(row["delta"]),
                float(row["alpha_pair"]),
                TWO_SIDED,
            )
            row["planned_n"] = int(planned_n)
            row["achieved_power_at_planned_n"] = planned_power
            row["underpowered_at_planned_n"] = bool(planned_power < float(row["power"]))
            rows.append(row)
        return rows

    out = table.copy()
    out["planned_n"] = int(planned_n)
    out["achieved_power_at_planned_n"] = [
        paired_t_power(
            int(planned_n),
            float(row.pilot_sd_diff),
            float(row.delta),
            float(row.alpha_pair),
            TWO_SIDED,
        )
        for row in out.itertuples(index=False)
    ]
    out["underpowered_at_planned_n"] = out["achieved_power_at_planned_n"] < out["power"]
    return out



def repeated_measures_anova(samples_by_method: dict[str, np.ndarray], metric: str, sample_unit: str) -> dict:
    names, y = sample_matrix(samples_by_method)
    n_subjects, n_methods = y.shape
    grand = float(np.mean(y))
    subject_means = np.mean(y, axis=1)
    method_means = np.mean(y, axis=0)
    ss_total = float(np.sum((y - grand) ** 2))
    ss_subject = float(n_methods * np.sum((subject_means - grand) ** 2))
    ss_method = float(n_subjects * np.sum((method_means - grand) ** 2))
    ss_error = max(0.0, ss_total - ss_subject - ss_method)
    df_method = n_methods - 1
    df_error = (n_subjects - 1) * (n_methods - 1)
    ms_method = ss_method / df_method if df_method > 0 else float("nan")
    ms_error = ss_error / df_error if df_error > 0 else float("nan")
    f_stat = ms_method / ms_error if ms_error > 0 else float("inf")
    p_value = float(stats.f.sf(f_stat, df_method, df_error)) if math.isfinite(f_stat) else 0.0
    return {
        "metric": metric,
        "sample_unit": sample_unit,
        "test": "repeated_measures_anova",
        "n_blocks": int(n_subjects),
        "n_algorithms": int(n_methods),
        "statistic": float(f_stat),
        "df1": int(df_method),
        "df2": int(df_error),
        "p_value": p_value,
    }


def friedman_multiple_sample_test(samples_by_method: dict[str, np.ndarray], metric: str, sample_unit: str) -> dict:
    names, y = sample_matrix(samples_by_method)
    statistic, p_value = stats.friedmanchisquare(*[y[:, i] for i in range(y.shape[1])])
    return {
        "metric": metric,
        "sample_unit": sample_unit,
        "test": "friedman",
        "n_blocks": int(y.shape[0]),
        "n_algorithms": int(y.shape[1]),
        "statistic": float(statistic),
        "df1": int(y.shape[1] - 1),
        "df2": None,
        "p_value": float(p_value),
    }


def global_multiple_sample_tests(metric_samples: dict[str, dict[str, np.ndarray]]):
    rows = []
    for metric, spec in metric_samples.items():
        samples = spec["samples"]
        sample_unit = spec["sample_unit"]
        rows.append(repeated_measures_anova(samples, metric, sample_unit))
        rows.append(friedman_multiple_sample_test(samples, metric, sample_unit))
    table = pd.DataFrame(rows) if pd is not None else rows
    if pd is not None:
        n_metric_tests = len(metric_samples)
        table["p_bonferroni_all_metrics"] = np.minimum(table["p_value"] * n_metric_tests, 1.0)
        table["reject_at_alpha"] = table["p_bonferroni_all_metrics"] < ALPHA
    return table


def posthoc_pairwise_table(metric_samples: dict[str, dict[str, np.ndarray]]):
    rows = []
    total_planned = 0
    for spec in metric_samples.values():
        total_planned += math.comb(len(spec["samples"]), 2)
    if FWER_SCOPE == "per_metric":
        total_planned = None

    for metric, spec in metric_samples.items():
        samples = spec["samples"]
        higher_is_better = bool(spec["higher_is_better"])
        names = list(samples)
        alpha_pair, scoped_total = pairwise_alpha_for_scope(len(names), ALPHA)
        correction_total = scoped_total if total_planned is None else total_planned
        for a, b in combinations(names, 2):
            a_values = np.asarray(samples[a], dtype=np.float64)
            b_values = np.asarray(samples[b], dtype=np.float64)
            diff = a_values - b_values
            t_stat, p_value = stats.ttest_rel(a_values, b_values)
            mean_diff = float(np.mean(diff))
            if mean_diff == 0.0:
                better = "tie"
            elif higher_is_better:
                better = a if mean_diff > 0.0 else b
            else:
                better = a if mean_diff < 0.0 else b
            rows.append({
                "metric": metric,
                "comparison": f"{a} vs {b}",
                "mean_diff_a_minus_b": mean_diff,
                "sd_diff": float(np.std(diff, ddof=1)),
                "t_statistic": float(t_stat),
                "p_value": float(p_value),
                "p_bonferroni": min(float(p_value) * correction_total, 1.0),
                "alpha_pair": float(alpha_pair),
                "fwer_scope": FWER_SCOPE,
                "total_planned_comparisons": int(correction_total),
                "significant": bool(float(p_value) < alpha_pair),
                "better_if_significant": better,
            })
    table = pd.DataFrame(rows) if pd is not None else rows
    if pd is not None:
        return table.sort_values(["metric", "p_bonferroni"]).reset_index(drop=True)
    return table


In [ ]:
pilot_sd_n = pilot_sample_size_for_sd(confidence=0.95, relative_error=0.25)
print(f"Pilot-study approximation for SD estimate: n ~= {pilot_sd_n} for 95% confidence and 25% relative error")
print(f"Using PILOT_QUALITY_N = {PILOT_QUALITY_N}")
print(f"Using PILOT_TIMING_REPEATS = {PILOT_TIMING_REPEATS} on TIMING_BATCH_N = {TIMING_BATCH_N}")

quality_pilot_x = rng.standard_normal((PILOT_QUALITY_N, DIM)).astype(np.float32)
timing_pilot_x = rng.standard_normal((TIMING_BATCH_N, DIM)).astype(np.float32)
quantizer_items, qtip_status = make_quantizer_items()
print(f"QTIP status: {qtip_status}")
print("Algorithms:")
for item in quantizer_items:
    print(f"- {item['method']} | rate={item['rate']:.3f} bpd")

pilot_reports = []
pilot_quality_scores = {}
for item in quantizer_items:
    report, recon, row_sqnr_bits = run_reconstruction(item, quality_pilot_x)
    pilot_reports.append(report)
    pilot_quality_scores[item["method"]] = row_sqnr_bits

pilot_timing_samples = benchmark_timing_samples(
    quantizer_items,
    timing_pilot_x,
    repeats=PILOT_TIMING_REPEATS,
    warmup=TIMING_WARMUP_REPEATS,
)

pilot_results = reports_table(pilot_reports)
quality_sample_size_table = sample_size_table_from_samples(
    pilot_quality_scores,
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_SQNR_BITS_DIFF,
    metric="per_vector_sqnr_bits",
    sample_unit="vectors",
)
quantize_time_sample_size_table = sample_size_table_from_samples(
    pilot_timing_samples["quantize_seconds"],
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_QUANTIZE_SECONDS,
    metric="quantize_seconds",
    sample_unit="timing_repeats",
)
dequantize_time_sample_size_table = sample_size_table_from_samples(
    pilot_timing_samples["dequantize_seconds"],
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_DEQUANTIZE_SECONDS,
    metric="dequantize_seconds",
    sample_unit="timing_repeats",
)

REQUIRED_QUALITY_N = required_n_from_table(quality_sample_size_table)
REQUIRED_QUANTIZE_REPEATS = required_n_from_table(quantize_time_sample_size_table)
REQUIRED_DEQUANTIZE_REPEATS = required_n_from_table(dequantize_time_sample_size_table)
EXPERIMENT_N = int(REQUIRED_QUALITY_N)
TIMING_REPEATS = int(max(REQUIRED_QUANTIZE_REPEATS, REQUIRED_DEQUANTIZE_REPEATS))

quality_sample_size_table = add_planned_power(quality_sample_size_table, EXPERIMENT_N)
quantize_time_sample_size_table = add_planned_power(quantize_time_sample_size_table, TIMING_REPEATS)
dequantize_time_sample_size_table = add_planned_power(dequantize_time_sample_size_table, TIMING_REPEATS)

sample_size_summary = pd.DataFrame([
    {
        "metric": "quality_per_vector_sqnr_bits",
        "sample_unit": "vectors",
        "pilot_samples": PILOT_QUALITY_N,
        "required_samples": REQUIRED_QUALITY_N,
        "planned_samples": EXPERIMENT_N,
        "minimum_interesting_difference": MIN_INTERESTING_SQNR_BITS_DIFF,
    },
    {
        "metric": "quantize_seconds",
        "sample_unit": "timing_repeats",
        "pilot_samples": PILOT_TIMING_REPEATS,
        "required_samples": REQUIRED_QUANTIZE_REPEATS,
        "planned_samples": TIMING_REPEATS,
        "minimum_interesting_difference": MIN_INTERESTING_QUANTIZE_SECONDS,
    },
    {
        "metric": "dequantize_seconds",
        "sample_unit": "timing_repeats",
        "pilot_samples": PILOT_TIMING_REPEATS,
        "required_samples": REQUIRED_DEQUANTIZE_REPEATS,
        "planned_samples": TIMING_REPEATS,
        "minimum_interesting_difference": MIN_INTERESTING_DEQUANTIZE_SECONDS,
    },
])

print(f"Pairwise comparisons: {math.comb(len(quantizer_items), 2)}")
alpha_pair, total_pairwise_tests = pairwise_alpha_for_scope(len(quantizer_items), ALPHA)
print(f"FWER scope: {FWER_SCOPE}; planned pairwise tests in correction: {total_pairwise_tests}")
print(f"Bonferroni alpha per comparison: {alpha_pair:.6g}")
print(f"Required quality vectors: {REQUIRED_QUALITY_N}")
print(f"Recommended quantize timing repeats: {REQUIRED_QUANTIZE_REPEATS}")
print(f"Recommended dequantize timing repeats: {REQUIRED_DEQUANTIZE_REPEATS}")
print(f"Required timing repeats: {TIMING_REPEATS}")
sample_size_summary


## Final experiment

The final quality run uses `EXPERIMENT_N` vectors from the quality sample-size calculation. Timing uses `TIMING_REPEATS` repeated trials on a fixed `TIMING_BATCH_N` batch. No maximum cap is applied; if the required timing repeat count is large, the final run may take a while.

In [ ]:
X = rng.standard_normal((EXPERIMENT_N, DIM)).astype(np.float32)
timing_x = rng.standard_normal((TIMING_BATCH_N, DIM)).astype(np.float32)
reports = []
reconstructions = {}
quality_scores = {}

for item in quantizer_items:
    report, recon, row_sqnr_bits = run_reconstruction(item, X)
    reports.append(report)
    reconstructions[report.method] = recon
    quality_scores[report.method] = row_sqnr_bits

final_timing_samples = benchmark_timing_samples(
    quantizer_items,
    timing_x,
    repeats=TIMING_REPEATS,
    warmup=TIMING_WARMUP_REPEATS,
)

final_table = reports_table(reports)
final_quality_sample_size_check = add_planned_power(sample_size_table_from_samples(
    quality_scores,
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_SQNR_BITS_DIFF,
    metric="per_vector_sqnr_bits",
    sample_unit="vectors",
), EXPERIMENT_N)
final_quantize_time_sample_size_check = add_planned_power(sample_size_table_from_samples(
    final_timing_samples["quantize_seconds"],
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_QUANTIZE_SECONDS,
    metric="quantize_seconds",
    sample_unit="timing_repeats",
), TIMING_REPEATS)
final_dequantize_time_sample_size_check = add_planned_power(sample_size_table_from_samples(
    final_timing_samples["dequantize_seconds"],
    alpha=ALPHA,
    power=POWER,
    delta=MIN_INTERESTING_DEQUANTIZE_SECONDS,
    metric="dequantize_seconds",
    sample_unit="timing_repeats",
), TIMING_REPEATS)

final_timing_summary = pd.DataFrame([
    {
        "method": name,
        "quantize_seconds_mean": float(np.mean(final_timing_samples["quantize_seconds"][name])),
        "quantize_seconds_std": float(np.std(final_timing_samples["quantize_seconds"][name], ddof=1)),
        "dequantize_seconds_mean": float(np.mean(final_timing_samples["dequantize_seconds"][name])),
        "dequantize_seconds_std": float(np.std(final_timing_samples["dequantize_seconds"][name], ddof=1)),
        "timing_repeats": TIMING_REPEATS,
        "timing_batch_n": TIMING_BATCH_N,
    }
    for name in final_timing_samples["quantize_seconds"]
])

metric_samples = {
    "sqnr_bits": {
        "samples": quality_scores,
        "sample_unit": "vectors",
        "higher_is_better": True,
    },
    "quantize_seconds": {
        "samples": final_timing_samples["quantize_seconds"],
        "sample_unit": "timing_repeats",
        "higher_is_better": False,
    },
    "dequantize_seconds": {
        "samples": final_timing_samples["dequantize_seconds"],
        "sample_unit": "timing_repeats",
        "higher_is_better": False,
    },
}

global_test_table = global_multiple_sample_tests(metric_samples)
posthoc_table = posthoc_pairwise_table(metric_samples)

final_table


In [ ]:
# Multiple-sample tests and corrected post-hoc comparisons from the final run.
print("Global multiple-sample tests")
display(global_test_table)
print("Bonferroni-corrected post-hoc paired comparisons")
display(posthoc_table)
print("Quality sample-size check")
display(final_quality_sample_size_check)
print("Quantize-time sample-size check")
display(final_quantize_time_sample_size_check)
print("Dequantize-time sample-size check")
display(final_dequantize_time_sample_size_check)
print("Timing summary")
display(final_timing_summary)
